# 04 — Model Training & Evaluation
Train a class-balanced Logistic Regression baseline using a stratified holdout and evaluate precision, recall, and ROC-AUC.

In [ ]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, roc_auc_score
df = pd.read_csv(Path('../data/raw/employee_attrition.csv'))
y = df.pop('Attrition').map({'No':0,'Yes':1})
X = df.drop(columns=['EmployeeNumber','EmployeeCount','Over18','StandardHours'])
cat = X.select_dtypes(include=['object','string']).columns
num = X.select_dtypes(exclude=['object','string']).columns
prep = ColumnTransformer([('num',StandardScaler(),num),('cat',OneHotEncoder(handle_unknown='ignore'),cat)])
pipe = Pipeline([('prep',prep),('model',LogisticRegression(class_weight='balanced',max_iter=2000))])
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.2,stratify=y,random_state=42)
pipe.fit(X_train,y_train)
p=pipe.predict(X_test); proba=pipe.predict_proba(X_test)[:,1]
print({'accuracy':round(accuracy_score(y_test,p),3),'precision':round(precision_score(y_test,p),3),'recall':round(recall_score(y_test,p),3),'roc_auc':round(roc_auc_score(y_test,proba),3)})